<a href="https://colab.research.google.com/github/marcocmcmelo/asistente_virtual_groq_whisper/blob/main/Assistente_de_voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Escolhendo a linguagem utilizada.

In [ ]:
language = 'pt'

## 2. Gravação de Áudio Com Python (e JavaScript)

In [ ]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

##

## 3. Integração com API do Groq

In [ ]:
!pip install groq

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "INSIRA AQUI SUA API"

## 4. Reconhecimento de Fala com Whisper (GROQ)

In [ ]:
from groq import Groq

client = Groq()

with open(record_file, "rb") as file:
    transcription = client.audio.transcriptions.create(
        file=file,
        model="whisper-large-v3-turbo",  # rápido e preciso
        response_format="text"
    )

texto_transcrito = transcription
print("Transcrição:", texto_transcrito)

## 5. Respsta da IA ao que foi perguntado.

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # modelo atual (use listagem se necessário)
    messages=[
        {"role": "user", "content": texto_transcrito}
    ]
)

resposta_ia = response.choices[0].message.content
print("Resposta IA:", resposta_ia)

## 6 - Limpeza do texto para reproduzir o áudio.

In [ ]:
!pip install groq gtts

In [ ]:
import re

def limpar_texto(texto):
    texto = re.sub(r"\*+", "", texto)
    texto = re.sub(r"`+", "", texto)
    texto = texto.replace("\n", ". ")
    return texto.strip()

texto_limpo = limpar_texto(resposta_ia)

## 7. Sintetizando a Resposta do Groq Como Voz.

In [ ]:
!pip install pydub

import os
import re
from gtts import gTTS
from IPython.display import Audio, display
from pydub import AudioSegment

# Limpeza + melhoria de fala
def preparar_texto(texto):
    texto = re.sub(r"\*+", "", texto)
    texto = texto.replace("\n", ". ")
    texto = texto.replace(":", ". ")
    texto = texto.replace(";", ". ")

    # pausas mais naturais
    texto = texto.replace(".", "... ")
    texto = texto.replace(",", ", ")

    return texto.strip()

# Dividir texto sem quebrar frases
def dividir_texto(texto, limite=400):
    partes = []

    while len(texto) > limite:
        corte = texto[:limite]

        if "." in corte:
            corte = corte[:corte.rfind(".") + 1]
        elif "," in corte:
            corte = corte[:corte.rfind(",") + 1]

        partes.append(corte.strip())
        texto = texto[len(corte):]

    if texto.strip():
        partes.append(texto.strip())

    return partes


# Garante texto
if 'texto_limpo' not in globals():
    if 'chatgpt_response' in globals():
        texto_limpo = preparar_texto(chatgpt_response)
    else:
        texto_limpo = "Olá. Este é um teste de áudio mais natural em português do Brasil."

# Gerar partes
arquivos = []
partes = dividir_texto(texto_limpo, limite=400)

for i, parte in enumerate(partes):
    nome = f"parte_{i}.mp3"

    try:
        tts = gTTS(text=parte, lang="pt", slow=False)
        tts.save(nome)
        arquivos.append(nome)
        print(f"Parte {i} gerada")

    except Exception as e:
        print(f"Erro na parte {i}:", e)

# Juntar tudo em um único áudio
audio_final = AudioSegment.empty()

for arq in arquivos:
    if os.path.exists(arq):
        audio = AudioSegment.from_file(arq)
        audio_final += audio + AudioSegment.silent(duration=300)  # pausa entre partes

# Exportar áudio final
arquivo_final = "resposta_final.mp3"
audio_final.export(arquivo_final, format="mp3")

print("Áudio final gerado:", arquivo_final)

# Reproduzir áudio final
display(Audio(arquivo_final, autoplay=True))